# 🔎 Materi 4 — Pencarian Informasi pada Kumpulan Data
### Training NLP

Bayangkan perusahaan memiliki **ribuan dokumen SOP, laporan, dan panduan**. Bagaimana menemukan dokumen yang tepat dalam hitungan detik — seperti "Google", tetapi untuk dokumen internal?

Bidang ini disebut **Information Retrieval (IR)**. Kita bangun bertahap, dari yang paling sederhana sampai yang dipakai industri:

| Bagian | Yang Dipraktikkan |
|--------|-------------------|
| 4.1 | Menyiapkan koleksi dokumen internal |
| 4.2 | Pencarian kata kunci persis — dan kelemahannya |
| 4.3 | **Mesin pencari mini**: TF-IDF + cosine similarity |
| 4.4 | Meningkatkan akurasi dengan **stemming** (Sastrawi) |
| 4.5 | Matriks kemiripan: menemukan dokumen duplikat/mirip |
| 4.6 | Menuju **semantic search** & chatbot dokumen (RAG) |

In [ ]:
# ===== Persiapan =====
!pip install Sastrawi -q

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stemmer = StemmerFactory().create_stemmer()
print("Library siap digunakan ✅")

## 4.1 Koleksi Dokumen Internal

Kita gunakan 10 dokumen contoh (di dunia nyata: ribuan). Setiap elemen mewakili judul/isi ringkas satu dokumen:

In [ ]:
koleksi_dokumen = [
    "SOP penggantian oli mesin sepeda motor dilakukan setiap 4000 km atau 4 bulan",
    "Laporan produksi harian mencatat jumlah unit sepeda motor yang selesai dirakit",
    "Prosedur klaim garansi sparepart wajib menyertakan bukti pembelian resmi",
    "Panduan keselamatan kerja mewajibkan penggunaan APD di seluruh area produksi",
    "Jadwal servis berkala pelanggan diatur melalui sistem booking online AHASS",
    "Checklist inspeksi kualitas dilakukan sebelum unit motor dikirim ke dealer",
    "Panduan penanganan komplain pelanggan di jaringan bengkel resmi",
    "Instruksi kerja perakitan mesin pada lini produksi utama pabrik",
    "Kebijakan pengembalian sparepart yang rusak dalam masa garansi",
    "Materi pelatihan mekanik baru tentang diagnosis kerusakan mesin motor",
]
print(f"Koleksi berisi {len(koleksi_dokumen)} dokumen ✅")

## 4.2 Pencarian Kata Kunci Persis (dan Kelemahannya)

Cara paling mudah: cek apakah kata yang dicari **muncul persis** di dalam dokumen.

In [ ]:
def cari_kata_kunci(kata):
    """Mencari dokumen yang mengandung teks persis seperti yang diketik."""
    hasil = [d for d in koleksi_dokumen if kata.lower() in d.lower()]
    return hasil if hasil else ["(tidak ditemukan)"]

print("Cari 'oli':")
for h in cari_kata_kunci("oli"):
    print("  -", h)
print()
print("Cari 'ganti oli':")
for h in cari_kata_kunci("ganti oli"):
    print("  -", h)     # ← GAGAL!

⚠️ **Masalahnya terlihat:** pencarian `"ganti oli"` gagal, padahal jelas ada dokumen tentang **"penggantian oli"** — metode ini hanya mencocokkan **teks persis**, bukan makna. Pengguna harus menebak kata yang tepat. Tidak praktis.

## 4.3 Mesin Pencari Mini — TF-IDF + Cosine Similarity

**Cara kerjanya** (fondasinya sudah kita pelajari di Materi 2):

```
Pertanyaan (query) ──► ubah jadi vektor angka ──► bandingkan dengan vektor semua
                        (TF-IDF)                   dokumen (cosine similarity, 0–1)
                                                          │
                              urutkan dari skor tertinggi ◄┘ ──► tampilkan hasil terbaik
```

In [ ]:
# Langkah 1: latih vectorizer pada seluruh koleksi (cukup sekali)
vectorizer = TfidfVectorizer()
matriks_dokumen = vectorizer.fit_transform(koleksi_dokumen)

def cari_dokumen(query, top_n=3):
    """Mengembalikan dokumen paling relevan beserta skor kemiripannya."""
    vektor_query = vectorizer.transform([query])                       # query → angka
    skor = cosine_similarity(vektor_query, matriks_dokumen).flatten()  # bandingkan
    urutan = skor.argsort()[::-1][:top_n]                              # ranking
    return pd.DataFrame({
        "Skor": skor[urutan].round(3),
        "Dokumen": [koleksi_dokumen[i] for i in urutan],
    })

cari_dokumen("jadwal ganti oli motor")

In [ ]:
# Coba beberapa pertanyaan lain — silakan ganti dengan pertanyaan Anda sendiri!
for q in ["cara klaim garansi", "aturan APD keselamatan", "pelatihan mekanik"]:
    print(f"Query: '{q}'")
    display(cari_dokumen(q))

✅ Jauh lebih baik daripada pencarian kata persis — dokumen relevan muncul **beserta skor relevansinya**. Tetapi masih ada celah: kata *ganti* (pada query) dan *penggantian* (pada dokumen) dianggap **dua kata berbeda**, sehingga skornya belum maksimal.

## 4.4 Meningkatkan Akurasi dengan Stemming

Solusinya: **stem dulu** semua dokumen dan query ke kata dasar (Materi 2), sehingga *ganti / penggantian / mengganti* menjadi kata yang sama: `ganti`.

In [ ]:
# Stem seluruh koleksi (dilakukan SEKALI di awal, hasilnya disimpan)
koleksi_stem = [stemmer.stem(d.lower()) for d in koleksi_dokumen]

print("Contoh sebelum:", koleksi_dokumen[0])
print("Contoh sesudah:", koleksi_stem[0])

In [ ]:
# Bangun mesin pencari versi 2: dengan stemming
vectorizer_v2 = TfidfVectorizer()
matriks_v2 = vectorizer_v2.fit_transform(koleksi_stem)

def cari_dokumen_v2(query, top_n=3):
    """Mesin pencari dengan stemming: query & dokumen disamakan ke kata dasar."""
    query_stem = stemmer.stem(query.lower())
    vektor_query = vectorizer_v2.transform([query_stem])
    skor = cosine_similarity(vektor_query, matriks_v2).flatten()
    urutan = skor.argsort()[::-1][:top_n]
    return pd.DataFrame({
        "Skor": skor[urutan].round(3),
        "Dokumen": [koleksi_dokumen[i] for i in urutan],   # tampilkan teks ASLI agar mudah dibaca
    })

query = "jadwal ganti oli motor"
print("=== TANPA stemming ===")
display(cari_dokumen(query))
print("=== DENGAN stemming ===")
display(cari_dokumen_v2(query))

📈 **Bandingkan skornya:** dengan stemming, dokumen *"SOP penggantian oli..."* mendapat skor lebih tinggi karena *ganti* (query) kini cocok dengan *penggantian* (dokumen). Satu tahap preprocessing → hasil pencarian nyata lebih baik.

## 4.5 Matriks Kemiripan — Menemukan Dokumen yang Mirip/Duplikat

Cosine similarity juga bisa membandingkan **dokumen dengan dokumen**. Kegunaannya: mendeteksi SOP ganda, mengelompokkan dokumen setopik, atau merekomendasikan "dokumen terkait".

In [ ]:
kemiripan_antar_dok = cosine_similarity(matriks_v2)

plt.figure(figsize=(8, 6.5))
plt.imshow(kemiripan_antar_dok, cmap="Reds", vmin=0, vmax=1)
plt.colorbar(label="skor kemiripan (0–1)")
label = [f"D{i+1}" for i in range(len(koleksi_dokumen))]
plt.xticks(range(len(label)), label)
plt.yticks(range(len(label)), label)
plt.title("Matriks Kemiripan Antar Dokumen")
plt.tight_layout(); plt.show()

# Cari pasangan dokumen paling mirip (di luar diagonal)
mat = kemiripan_antar_dok.copy()
np.fill_diagonal(mat, 0)
i, j = np.unravel_index(mat.argmax(), mat.shape)
print(f"Pasangan paling mirip (skor {mat[i, j]:.2f}):")
print(f"  D{i+1}: {koleksi_dokumen[i]}")
print(f"  D{j+1}: {koleksi_dokumen[j]}")

## 4.6 Menuju Semantic Search & Chatbot Dokumen (RAG)

Yang baru kita bangun adalah tangga menuju sistem pencarian modern:

| Level | Teknologi | Kemampuan |
|-------|-----------|-----------|
| 1 | Kata kunci persis | Cocok teks sama persis saja |
| 2 | **TF-IDF + cosine** *(praktik kita)* | Bobot kata + skor relevansi |
| 3 | + **Stemming** *(praktik kita)* | Variasi imbuhan ikut cocok |
| 4 | **Semantic search** (embedding) | Paham **makna**: *"motor tidak bisa distarter"* menemukan *"troubleshooting sistem pengapian"* |
| 5 | **RAG** (Retrieval-Augmented Generation) | Semantic search + LLM → **chatbot yang menjawab dari dokumen internal** |

💡 **Prinsip semua level tetap sama** dengan yang kita bangun hari ini: *teks → vektor → hitung kemiripan → ranking*. Yang berubah hanyalah cara mengubah teks menjadi vektor.

---
# 🎯 Rangkuman & Latihan Mandiri — Materi 4

| Teknik | Fungsi yang Kita Buat | Kegunaan |
|--------|----------------------|----------|
| Keyword search | `cari_kata_kunci()` | Cepat, tapi rapuh |
| TF-IDF + cosine | `cari_dokumen()` | Mesin pencari dengan skor relevansi |
| + Stemming | `cari_dokumen_v2()` | Akurasi lebih tinggi untuk Bahasa Indonesia |
| Matriks kemiripan | `cosine_similarity(matriks)` | Deteksi duplikat & dokumen terkait |

### ✍️ Latihan Mandiri
1. Tambahkan 3 dokumen SOP dari unit kerja Anda ke `koleksi_dokumen`, lalu jalankan ulang **seluruh** sel di bawahnya (vectorizer harus dilatih ulang!).
2. Coba 5 query berbeda pada `cari_dokumen_v2()` — catat query mana yang hasilnya kurang tepat, dan pikirkan mengapa.
3. Ubah `top_n` menjadi 5 — apakah hasil urutan 4–5 masih relevan?
4. **Tantangan:** buat fungsi `dokumen_terkait(nomor_dokumen)` yang mengembalikan 2 dokumen paling mirip dengan dokumen tersebut (gunakan `kemiripan_antar_dok`).

🎉 **Selamat!** Anda telah menyelesaikan seluruh rangkaian praktik: dari ekstraksi data, memahami cara AI membaca teks, mengambil poin penting, hingga membangun mesin pencari mini. Fondasi ini adalah pintu masuk menuju otomasi dokumen dan penerapan AI di proses kerja Anda. 🚀

---
## 💡 Jawaban Latihan Mandiri

### Jawaban 1 - Menambahkan 3 Dokumen SOP + Melatih Ulang Model

Sesuai instruksi, kita menambah 3 dokumen lalu melatih ulang TF-IDF (versi biasa dan versi stemming).

In [ ]:
# Latihan 1: tambah 3 dokumen SOP baru
dokumen_tambahan = [
    "SOP penanganan keluhan pelanggan melalui call center wajib ditutup maksimal 2x24 jam",
    "Panduan troubleshooting motor tidak bisa distarter dimulai dari pengecekan aki dan busi",
    "Prosedur kalibrasi alat ukur torsi dilakukan setiap awal bulan oleh teknisi quality assurance",
]

koleksi_dokumen.extend(dokumen_tambahan)

# WAJIB: latih ulang karena koleksi berubah
vectorizer = TfidfVectorizer()
matriks_dokumen = vectorizer.fit_transform(koleksi_dokumen)

koleksi_stem = [stemmer.stem(d.lower()) for d in koleksi_dokumen]
vectorizer_v2 = TfidfVectorizer()
matriks_v2 = vectorizer_v2.fit_transform(koleksi_stem)
kemiripan_antar_dok = cosine_similarity(matriks_v2)

print(f"Total dokumen setelah ditambah: {len(koleksi_dokumen)}")
for i, d in enumerate(dokumen_tambahan, start=1):
    print(f"Dokumen tambahan {i}: {d}")

**Uraian:**
- Kenapa harus latih ulang? Karena vocabulary TF-IDF berubah saat dokumen baru masuk.
- Jika tidak latih ulang, kata-kata baru (misalnya *distarter*, *kalibrasi*) tidak masuk model.

### Jawaban 2 - Uji 5 Query pada `cari_dokumen_v2()`

In [ ]:
# Latihan 2: uji 5 query berbeda
query_uji = [
    "cara klaim garansi sparepart",
    "aturan keselamatan APD di pabrik",
    "motor tidak bisa distarter",
    "jadwal servis online pelanggan",
    "tips hemat bensin motor",
]

for q in query_uji:
    print(f"\nQuery: {q}")
    display(cari_dokumen_v2(q, top_n=3))

**Uraian hasil (contoh analisis):**
- Query **"cara klaim garansi sparepart"**: hasil umumnya tepat karena kata kunci sangat spesifik.
- Query **"aturan keselamatan APD di pabrik"**: hasil utama tepat ke dokumen APD/keselamatan.
- Query **"motor tidak bisa distarter"**: setelah dokumen troubleshooting ditambahkan, hasil menjadi lebih relevan.
- Query **"jadwal servis online pelanggan"**: biasanya cocok ke dokumen booking AHASS.
- Query **"tips hemat bensin motor"**: cenderung kurang tepat karena topik ini belum ada di koleksi dokumen (out-of-scope), sehingga mesin tetap memaksa memilih dokumen paling dekat secara kata.

### Jawaban 3 - Ubah `top_n` Menjadi 5

In [ ]:
# Latihan 3: lihat dampak top_n = 5
query_top5 = "jadwal ganti oli motor"
hasil_top5 = cari_dokumen_v2(query_top5, top_n=5)
print(f"Query: {query_top5}")
hasil_top5

**Uraian:**
- Biasanya urutan 1-3 masih relevan tinggi (kata inti masih sama).
- Urutan 4-5 sering mulai turun relevansinya; kadang hanya berbagi kata umum seperti *motor* atau *prosedur*.
- Ini normal di IR klasik: semakin ke bawah ranking, sinyal kata makin lemah.

### Jawaban 4 (Tantangan) - Fungsi `dokumen_terkait(nomor_dokumen)`

In [ ]:
# Latihan 4 (tantangan)
def dokumen_terkait(nomor_dokumen):
    """Kembalikan 2 dokumen paling mirip dari nomor dokumen (format input: 1..N)."""
    idx = nomor_dokumen - 1

    if idx < 0 or idx >= len(koleksi_dokumen):
        raise ValueError(f"nomor_dokumen harus di antara 1 sampai {len(koleksi_dokumen)}")

    skor = kemiripan_antar_dok[idx].copy()
    skor[idx] = -1  # abaikan dokumen itu sendiri
    urutan = skor.argsort()[::-1][:2]

    return pd.DataFrame({
        "Dokumen_Acuan": [f"D{nomor_dokumen}"] * 2,
        "Dokumen_Terkait": [f"D{i+1}" for i in urutan],
        "Skor": skor[urutan].round(3),
        "Isi_Dokumen": [koleksi_dokumen[i] for i in urutan],
    })

# Contoh pemakaian
print("Dokumen acuan: D1")
display(dokumen_terkait(1))

**Uraian:**
- Fungsi mengambil 1 baris dari `kemiripan_antar_dok` (dokumen acuan vs semua dokumen).
- Nilai diagonal (dokumen dengan dirinya sendiri) diabaikan agar tidak terpilih.
- Hasil diurutkan dari skor tertinggi lalu diambil 2 teratas sebagai dokumen terkait.